In [ ]:
!pip install flask

In [ ]:
import os

os.makedirs("templates", exist_ok=True)
os.makedirs("static", exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


In [ ]:
from flask import Flask, render_template, request, redirect, url_for, flash
import sqlite3
import os

app = Flask(__name__)
app.secret_key = "library_secret_key"

DATABASE = "library.db"


# ---------------- DATABASE CONNECTION ----------------

def get_db():
    conn = sqlite3.connect(DATABASE)
    conn.row_factory = sqlite3.Row
    return conn


# ---------------- CREATE DATABASE ----------------

def create_database():

    conn = get_db()
    cursor = conn.cursor()

    # Books table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS books (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT NOT NULL,
        author TEXT NOT NULL,
        category TEXT NOT NULL,
        quantity INTEGER NOT NULL,
        available INTEGER NOT NULL
    )
    """)

    # Members table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS members (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        email TEXT NOT NULL,
        phone TEXT NOT NULL,
        department TEXT NOT NULL
    )
    """)

    # Issue table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS issued_books (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        book_id INTEGER,
        member_id INTEGER,
        issue_date TEXT NOT NULL,
        return_date TEXT,
        status TEXT NOT NULL
    )
    """)

    # Add sample books only if database is empty
    cursor.execute("SELECT COUNT(*) FROM books")
    book_count = cursor.fetchone()[0]

    if book_count == 0:
        sample_books = [
            ("Python Programming", "Mark Lutz", "Programming", 5, 5),
            ("Data Structures", "Seymour Lipschutz", "Computer Science", 4, 4),
            ("Database Management", "Raghu Ramakrishnan", "Database", 3, 3),
            ("Web Technology", "Uttam Kumar Roy", "Web Development", 5, 5),
            ("Artificial Intelligence", "Stuart Russell", "AI", 3, 3)
        ]

        cursor.executemany("""
        INSERT INTO books
        (title, author, category, quantity, available)
        VALUES (?, ?, ?, ?, ?)
        """, sample_books)

    # Add sample members
    cursor.execute("SELECT COUNT(*) FROM members")
    member_count = cursor.fetchone()[0]

    if member_count == 0:
        sample_members = [
            ("Shagun Dudhmogre", "shagun@example.com", "9876543210", "Information Technology"),
            ("Rahul Sharma", "rahul@example.com", "9876543211", "Computer Science"),
            ("Priya Verma", "priya@example.com", "9876543212", "Information Technology")
        ]

        cursor.executemany("""
        INSERT INTO members
        (name, email, phone, department)
        VALUES (?, ?, ?, ?)
        """, sample_members)

    conn.commit()
    conn.close()


# ---------------- HOME PAGE ----------------

@app.route("/")
def index():

    conn = get_db()

    total_books = conn.execute(
        "SELECT COUNT(*) FROM books"
    ).fetchone()[0]

    total_members = conn.execute(
        "SELECT COUNT(*) FROM members"
    ).fetchone()[0]

    issued_books = conn.execute(
        "SELECT COUNT(*) FROM issued_books WHERE status='Issued'"
    ).fetchone()[0]

    available_books = conn.execute(
        "SELECT SUM(available) FROM books"
    ).fetchone()[0]

    if available_books is None:
        available_books = 0

    conn.close()

    return render_template(
        "index.html",
        total_books=total_books,
        total_members=total_members,
        issued_books=issued_books,
        available_books=available_books
    )


# ---------------- BOOKS PAGE ----------------

@app.route("/books")
def books():

    search = request.args.get("search", "")

    conn = get_db()

    if search:
        books = conn.execute("""
        SELECT * FROM books
        WHERE title LIKE ?
        OR author LIKE ?
        OR category LIKE ?
        """,
        (
            "%" + search + "%",
            "%" + search + "%",
            "%" + search + "%"
        )).fetchall()
    else:
        books = conn.execute(
            "SELECT * FROM books ORDER BY id DESC"
        ).fetchall()

    conn.close()

    return render_template(
        "books.html",
        books=books,
        search=search
    )


# ---------------- ADD BOOK ----------------

@app.route("/add_book", methods=["POST"])
def add_book():

    title = request.form["title"]
    author = request.form["author"]
    category = request.form["category"]
    quantity = int(request.form["quantity"])

    conn = get_db()

    conn.execute("""
    INSERT INTO books
    (title, author, category, quantity, available)
    VALUES (?, ?, ?, ?, ?)
    """,
    (title, author, category, quantity, quantity))

    conn.commit()
    conn.close()

    return redirect(url_for("books"))


# ---------------- DELETE BOOK ----------------

@app.route("/delete_book/<int:id>")
def delete_book(id):

    conn = get_db()

    conn.execute(
        "DELETE FROM books WHERE id=?",
        (id,)
    )

    conn.commit()
    conn.close()

    return redirect(url_for("books"))


# ---------------- MEMBERS PAGE ----------------

@app.route("/members")
def members():

    conn = get_db()

    members = conn.execute(
        "SELECT * FROM members ORDER BY id DESC"
    ).fetchall()

    conn.close()

    return render_template(
        "members.html",
        members=members
    )


# ---------------- ADD MEMBER ----------------

@app.route("/add_member", methods=["POST"])
def add_member():

    name = request.form["name"]
    email = request.form["email"]
    phone = request.form["phone"]
    department = request.form["department"]

    conn = get_db()

    conn.execute("""
    INSERT INTO members
    (name, email, phone, department)
    VALUES (?, ?, ?, ?)
    """,
    (name, email, phone, department))

    conn.commit()
    conn.close()

    return redirect(url_for("members"))


# ---------------- DELETE MEMBER ----------------

@app.route("/delete_member/<int:id>")
def delete_member(id):

    conn = get_db()

    conn.execute(
        "DELETE FROM members WHERE id=?",
        (id,)
    )

    conn.commit()
    conn.close()

    return redirect(url_for("members"))


# ---------------- ISSUE BOOK PAGE ----------------

@app.route("/issue")
def issue():

    conn = get_db()

    books = conn.execute(
        "SELECT * FROM books WHERE available > 0"
    ).fetchall()

    members = conn.execute(
        "SELECT * FROM members"
    ).fetchall()

    issued = conn.execute("""
    SELECT issued_books.id,
           books.title,
           members.name,
           issued_books.issue_date,
           issued_books.return_date,
           issued_books.status
    FROM issued_books
    JOIN books ON issued_books.book_id = books.id
    JOIN members ON issued_books.member_id = members.id
    ORDER BY issued_books.id DESC
    """).fetchall()

    conn.close()

    return render_template(
        "issue.html",
        books=books,
        members=members,
        issued=issued
    )


# ---------------- ISSUE BOOK ----------------

@app.route("/issue_book", methods=["POST"])
def issue_book():

    book_id = request.form["book_id"]
    member_id = request.form["member_id"]
    issue_date = request.form["issue_date"]

    conn = get_db()

    book = conn.execute(
        "SELECT available FROM books WHERE id=?",
        (book_id,)
    ).fetchone()

    if book and book["available"] > 0:

        conn.execute("""
        INSERT INTO issued_books
        (book_id, member_id, issue_date, status)
        VALUES (?, ?, ?, 'Issued')
        """,
        (book_id, member_id, issue_date))

        conn.execute("""
        UPDATE books
        SET available = available - 1
        WHERE id=?
        """,
        (book_id,))

        conn.commit()

    conn.close()

    return redirect(url_for("issue"))


# ---------------- RETURN BOOK ----------------

@app.route("/return_book/<int:id>")
def return_book(id):

    conn = get_db()

    record = conn.execute("""
    SELECT book_id
    FROM issued_books
    WHERE id=?
    """, (id,)).fetchone()

    if record:

        conn.execute("""
        UPDATE issued_books
        SET status='Returned'
        WHERE id=?
        """, (id,))

        conn.execute("""
        UPDATE books
        SET available = available + 1
        WHERE id=?
        """, (record["book_id"],))

        conn.commit()

    conn.close()

    return redirect(url_for("issue"))


# ---------------- START APPLICATION ----------------

create_database()

print("Library Management System is ready!")

Library Management System is ready!


In [ ]:
%%writefile templates/index.html

<!DOCTYPE html>
<html>

<head>

    <title>Library Management System</title>

    <meta name="viewport" content="width=device-width, initial-scale=1.0">

    <link rel="stylesheet"
    href="{{ url_for('static', filename='style.css') }}">

</head>

<body>

<div class="sidebar">

    <h2>📚 MyLibrary</h2>

    <a href="/">🏠 Dashboard</a>
    <a href="/books">📖 Books</a>
    <a href="/members">👨‍🎓 Members</a>
    <a href="/issue">🔄 Issue / Return</a>

</div>


<div class="main">

    <div class="topbar">

        <div>
            <h1>Library Management System</h1>
            <p>Welcome to your digital library</p>
        </div>

        <div class="profile">
            👤 Admin
        </div>

    </div>


    <div class="cards">

        <div class="card">
            <div class="icon">📚</div>
            <h3>Total Books</h3>
            <h2>{{ total_books }}</h2>
        </div>

        <div class="card">
            <div class="icon">👨‍🎓</div>
            <h3>Total Members</h3>
            <h2>{{ total_members }}</h2>
        </div>

        <div class="card">
            <div class="icon">📕</div>
            <h3>Issued Books</h3>
            <h2>{{ issued_books }}</h2>
        </div>

        <div class="card">
            <div class="icon">✅</div>
            <h3>Available Books</h3>
            <h2>{{ available_books }}</h2>
        </div>

    </div>


    <div class="welcome">

        <h2>📚 Smart Library Management</h2>

        <p>
            Manage books, members and book transactions
            easily using this simple digital library system.
        </p>

        <div class="buttons">

            <a href="/books" class="btn">
                Manage Books
            </a>

            <a href="/members" class="btn">
                Manage Members
            </a>

            <a href="/issue" class="btn">
                Issue / Return
            </a>

        </div>

    </div>

</div>

</body>
</html>

Writing templates/index.html


In [ ]:
%%writefile templates/books.html

<!DOCTYPE html>
<html>

<head>

<title>Books - MyLibrary</title>

<link rel="stylesheet"
href="{{ url_for('static', filename='style.css') }}">

</head>

<body>

<div class="sidebar">

<h2>📚 MyLibrary</h2>

<a href="/">🏠 Dashboard</a>
<a href="/books">📖 Books</a>
<a href="/members">👨‍🎓 Members</a>
<a href="/issue">🔄 Issue / Return</a>

</div>


<div class="main">

<h1>📖 Book Management</h1>

<div class="form-box">

<h2>Add New Book</h2>

<form method="POST" action="/add_book">

<input
type="text"
name="title"
placeholder="Book Title"
required>

<input
type="text"
name="author"
placeholder="Author Name"
required>

<select name="category" required>

<option value="">Select Category</option>

<option>Programming</option>
<option>Computer Science</option>
<option>Database</option>
<option>Web Development</option>
<option>Artificial Intelligence</option>
<option>Mathematics</option>

</select>

<input
type="number"
name="quantity"
placeholder="Quantity"
min="1"
required>

<button type="submit">
➕ Add Book
</button>

</form>

</div>


<div class="table-box">

<div class="search">

<form method="GET">

<input
type="text"
name="search"
placeholder="🔍 Search books..."
value="{{ search }}">

<button type="submit">
Search
</button>

</form>

</div>


<table>

<tr>

<th>ID</th>
<th>Book</th>
<th>Author</th>
<th>Category</th>
<th>Quantity</th>
<th>Available</th>
<th>Action</th>

</tr>


{% for book in books %}

<tr>

<td>{{ book.id }}</td>

<td>{{ book.title }}</td>

<td>{{ book.author }}</td>

<td>{{ book.category }}</td>

<td>{{ book.quantity }}</td>

<td>

<span class="available">
{{ book.available }}
</span>

</td>

<td>

<a
class="delete"
href="/delete_book/{{ book.id }}"
onclick="return confirmDelete()">

Delete

</a>

</td>

</tr>

{% endfor %}

</table>

</div>

</div>

<script src="{{ url_for('static', filename='script.js') }}"></script>

</body>
</html>

Writing templates/books.html


In [ ]:
%%writefile templates/members.html

<!DOCTYPE html>
<html>

<head>

<title>Members - MyLibrary</title>

<link rel="stylesheet"
href="{{ url_for('static', filename='style.css') }}">

</head>

<body>

<div class="sidebar">

<h2>📚 MyLibrary</h2>

<a href="/">🏠 Dashboard</a>
<a href="/books">📖 Books</a>
<a href="/members">👨‍🎓 Members</a>
<a href="/issue">🔄 Issue / Return</a>

</div>


<div class="main">

<h1>👨‍🎓 Member Management</h1>


<div class="form-box">

<h2>Add New Member</h2>

<form method="POST" action="/add_member">

<input
type="text"
name="name"
placeholder="Member Name"
required>

<input
type="email"
name="email"
placeholder="Email Address"
required>

<input
type="text"
name="phone"
placeholder="Phone Number"
required>

<select name="department" required>

<option value="">Select Department</option>

<option>Information Technology</option>
<option>Computer Science</option>
<option>Electronics</option>
<option>Mechanical</option>
<option>Civil</option>

</select>

<button type="submit">
➕ Add Member
</button>

</form>

</div>


<div class="table-box">

<table>

<tr>

<th>ID</th>
<th>Name</th>
<th>Email</th>
<th>Phone</th>
<th>Department</th>
<th>Action</th>

</tr>


{% for member in members %}

<tr>

<td>{{ member.id }}</td>

<td>{{ member.name }}</td>

<td>{{ member.email }}</td>

<td>{{ member.phone }}</td>

<td>{{ member.department }}</td>

<td>

<a
class="delete"
href="/delete_member/{{ member.id }}"
onclick="return confirmDelete()">

Delete

</a>

</td>

</tr>

{% endfor %}

</table>

</div>

</div>

<script src="{{ url_for('static', filename='script.js') }}"></script>

</body>
</html>

Writing templates/members.html


In [ ]:
%%writefile templates/issue.html

<!DOCTYPE html>
<html>

<head>

<title>Issue Books - MyLibrary</title>

<link rel="stylesheet"
href="{{ url_for('static', filename='style.css') }}">

</head>

<body>

<div class="sidebar">

<h2>📚 MyLibrary</h2>

<a href="/">🏠 Dashboard</a>
<a href="/books">📖 Books</a>
<a href="/members">👨‍🎓 Members</a>
<a href="/issue">🔄 Issue / Return</a>

</div>


<div class="main">

<h1>🔄 Issue / Return Books</h1>


<div class="form-box">

<h2>Issue a Book</h2>

<form method="POST" action="/issue_book">


<label>Select Book</label>

<select name="book_id" required>

<option value="">
Choose a book
</option>

{% for book in books %}

<option value="{{ book.id }}">

{{ book.title }} - Available: {{ book.available }}

</option>

{% endfor %}

</select>


<label>Select Member</label>

<select name="member_id" required>

<option value="">
Choose member
</option>

{% for member in members %}

<option value="{{ member.id }}">

{{ member.name }}

</option>

{% endfor %}

</select>


<label>Issue Date</label>

<input
type="date"
name="issue_date"
required>


<button type="submit">

📕 Issue Book

</button>

</form>

</div>


<div class="table-box">

<h2>Transaction History</h2>

<table>

<tr>

<th>ID</th>
<th>Book</th>
<th>Member</th>
<th>Issue Date</th>
<th>Return Date</th>
<th>Status</th>
<th>Action</th>

</tr>


{% for item in issued %}

<tr>

<td>{{ item.id }}</td>

<td>{{ item.title }}</td>

<td>{{ item.name }}</td>

<td>{{ item.issue_date }}</td>

<td>

{{ item.return_date or "-" }}

</td>

<td>

{% if item.status == "Issued" %}

<span class="issued">
Issued
</span>

{% else %}

<span class="returned">
Returned
</span>

{% endif %}

</td>

<td>

{% if item.status == "Issued" %}

<a
class="return-btn"
href="/return_book/{{ item.id }}">

Return

</a>

{% else %}

Done

{% endif %}

</td>

</tr>

{% endfor %}

</table>

</div>

</div>

</body>

</html>

Writing templates/issue.html


In [ ]:
%%writefile static/style.css

* {
    margin: 0;
    padding: 0;
    box-sizing: border-box;
    font-family: Arial, sans-serif;
}

body {
    background: #f4f7fb;
    color: #263238;
}


/* SIDEBAR */

.sidebar {
    position: fixed;
    left: 0;
    top: 0;

    width: 240px;
    height: 100vh;

    background: linear-gradient(
        180deg,
        #172554,
        #1e3a8a,
        #2563eb
    );

    padding: 25px 15px;

    box-shadow: 3px 0 15px rgba(0,0,0,0.15);
}

.sidebar h2 {
    color: white;
    text-align: center;
    margin-bottom: 35px;
}

.sidebar a {
    display: block;

    color: white;
    text-decoration: none;

    padding: 14px 18px;
    margin: 10px 0;

    border-radius: 10px;

    transition: 0.3s;
}

.sidebar a:hover {
    background: rgba(255,255,255,0.18);
    transform: translateX(5px);
}


/* MAIN */

.main {
    margin-left: 240px;
    padding: 35px;
}

.main h1 {
    color: #172554;
    margin-bottom: 25px;
}


/* TOP BAR */

.topbar {
    display: flex;
    justify-content: space-between;
    align-items: center;

    margin-bottom: 30px;
}

.topbar p {
    color: #64748b;
}

.profile {
    background: white;
    padding: 12px 20px;
    border-radius: 25px;

    box-shadow: 0 4px 15px rgba(0,0,0,0.08);
}


/* CARDS */

.cards {
    display: grid;

    grid-template-columns:
    repeat(4, 1fr);

    gap: 20px;

    margin-bottom: 30px;
}

.card {
    background: white;

    padding: 25px;

    border-radius: 15px;

    box-shadow:
    0 5px 20px rgba(0,0,0,0.08);

    transition: 0.3s;
}

.card:hover {
    transform: translateY(-7px);
}

.card h3 {
    color: #64748b;
    margin: 10px 0;
}

.card h2 {
    color: #2563eb;
    font-size: 30px;
}

.icon {
    font-size: 35px;
}


/* WELCOME */

.welcome {
    background: white;

    padding: 35px;

    border-radius: 18px;

    box-shadow:
    0 5px 20px rgba(0,0,0,0.08);
}

.welcome h2 {
    color: #1e3a8a;
    margin-bottom: 15px;
}

.welcome p {
    color: #64748b;
    margin-bottom: 25px;
}


/* BUTTONS */

.buttons {
    display: flex;
    gap: 15px;
    flex-wrap: wrap;
}

.btn,
button {
    background: #2563eb;

    color: white;

    border: none;

    padding: 12px 20px;

    border-radius: 8px;

    text-decoration: none;

    cursor: pointer;

    transition: 0.3s;
}

.btn:hover,
button:hover {
    background: #1d4ed8;
    transform: translateY(-2px);
}


/* FORM */

.form-box {
    background: white;

    padding: 25px;

    border-radius: 15px;

    margin-bottom: 25px;

    box-shadow:
    0 5px 20px rgba(0,0,0,0.07);
}

.form-box h2 {
    color: #1e3a8a;
    margin-bottom: 20px;
}

form {
    display: flex;
    gap: 12px;
    flex-wrap: wrap;
}

input,
select {
    padding: 12px;

    border: 1px solid #dbe3ef;

    border-radius: 8px;

    outline: none;

    background: #f8fafc;

    min-width: 180px;
}

input:focus,
select:focus {
    border-color: #2563eb;
}


/* TABLE */

.table-box {
    background: white;

    padding: 25px;

    border-radius: 15px;

    box-shadow:
    0 5px 20px rgba(0,0,0,0.07);

    overflow-x: auto;
}

table {
    width: 100%;

    border-collapse: collapse;

    margin-top: 15px;
}

th {
    background: #1e3a8a;
    color: white;

    padding: 14px;

    text-align: left;
}

td {
    padding: 13px;

    border-bottom:
    1px solid #e5e7eb;
}

tr:hover {
    background: #f8fafc;
}


/* STATUS */

.available {
    color: #15803d;
    font-weight: bold;
}

.issued {
    background: #fef3c7;

    color: #92400e;

    padding: 6px 12px;

    border-radius: 20px;
}

.returned {
    background: #dcfce7;

    color: #166534;

    padding: 6px 12px;

    border-radius: 20px;
}


/* DELETE */

.delete {
    color: #dc2626;
    text-decoration: none;
    font-weight: bold;
}

.return-btn {
    background: #16a34a;

    color: white;

    padding: 7px 12px;

    border-radius: 6px;

    text-decoration: none;
}


/* SEARCH */

.search {
    margin-bottom: 15px;
}


/* MOBILE */

@media(max-width: 900px) {

    .sidebar {
        width: 190px;
    }

    .main {
        margin-left: 190px;
    }

    .cards {
        grid-template-columns:
        repeat(2, 1fr);
    }

}

@media(max-width: 600px) {

    .sidebar {
        position: relative;
        width: 100%;
        height: auto;
    }

    .main {
        margin-left: 0;
        padding: 20px;
    }

    .cards {
        grid-template-columns: 1fr;
    }

    .topbar {
        flex-direction: column;
        align-items: flex-start;
        gap: 15px;
    }

}

Writing static/style.css


In [ ]:
%%writefile static/script.js

function confirmDelete() {

    return confirm(
        "Are you sure you want to delete this record?"
    );

}


// Automatically set today's date
const dateInput =
    document.querySelector('input[name="issue_date"]');

if (dateInput) {

    const today =
        new Date().toISOString().split("T")[0];

    dateInput.value = today;

}

Writing static/script.js


In [ ]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(5000)")
print(url)

https://5000-m-s-kkb-usc1b1-2ba99b2gu7mju-b.us-central1-1.prod.colab.dev


In [ ]:
app.run(host="0.0.0.0", port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [07/Oct/2026 10:14:34] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Oct/2026 10:14:34] "GET /static/style.css HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Oct/2026 10:14:35] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [07/Oct/2026 10:15:47] "GET /books HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Oct/2026 10:15:47] "GET /static/style.css HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Oct/2026 10:15:47] "GET /static/script.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Oct/2026 10:15:52] "GET /members HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Oct/2026 10:15:52] "GET /static/script.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Oct/2026 